# Assignment 5 — Catching Data Leakage Before It Catches You

**Course:** Feature Engineering & MLOps  
**Assignment:** 5  
**Topic:** Data Leakage — Target Leakage and Preprocessing Leakage

# Assignment 5 — Requirements & Completion Checklist

> Based directly on the Assignment 5 PDF. The notebook below contains the required markdown sections, code, tables, and executed outputs from the supplied assignment notebook.

## Required tasks
### 4.1 Target leakage
- [x] Load the 600-customer telecom churn dataset.
- [x] Count non-null `days_since_cancellation` and verify in one line that it exactly equals the number of `churn == 1` rows.
- [x] Fill missing `final_bill_amount` with `-1` and correlate it with `churn`.
- [x] Compare that correlation with legitimate `tenure_months`.
- [x] Train Model A using only legitimate features: `tenure_months`, `monthly_charges`, `contract_type`, `support_calls`.
- [x] Report Model A test accuracy and ROC-AUC.
- [x] Train Model B using legitimate + leaked features: `days_since_cancellation`, `final_bill_amount`.
- [x] Report Model B test accuracy and ROC-AUC.
- [x] Report the accuracy and ROC-AUC gaps between Model B and Model A.
- [x] Explain in 2–3 sentences why Model B cannot work honestly for brand-new applicants.

### 4.2 Preprocessing leakage
- [x] Use only legitimate numeric features: `tenure_months`, `monthly_charges`, `support_calls`.
- [x] Fit `StandardScaler` on the entire dataset first and report `mean_` and `scale_`.
- [x] Split first, then fit a new scaler on the training split only and report `mean_` and `scale_`.
- [x] Report the numeric difference between the full-data and training-only scaler means.
- [x] Explain why preprocessing must not learn from held-out test data, even when the numerical difference is small.
- [x] Rebuild preprocessing as one sklearn `Pipeline` containing imputer + scaler + model.
- [x] Confirm the pipeline reproduces the manual training-only scaler statistics.

### 4.3 The fix
- [x] Use only legitimate predictors.
- [x] Split before preprocessing.
- [x] Fit preprocessing only through the training split.
- [x] Retrain the final model using the correctly ordered pipeline.
- [x] Report final test accuracy and ROC-AUC.
- [x] Confirm the final fixed model matches Model A.

### Optional bonus
- [x] Demonstrate cross-validation preprocessing leakage by scaling the entire dataset before 5-fold CV.
- [x] Compare against a `Pipeline` that fits `StandardScaler` inside each CV fold.
- [x] Explain the observed difference or why it may be small.

## Submission requirements
- [x] Notebook filename: `Assignment5_DataLeakage.ipynb`
- [x] Clear markdown headers matching Sections 4.1–4.3.
- [x] Outputs retained.
- [ ] Commit under the repository's top-level `Assignments` folder.
- [ ] Run top-to-bottom in the target repository environment and push to GitHub.
- [ ] Share the GitHub repository link with the instructor.

## Important dataset note
The current chat upload included the Assignment 5 PDF and completed notebook, but **did not include the raw `customer_churn_a5.csv` file**. Therefore the supplied notebook's cached outputs were preserved rather than inventing/reconstructing a different dataset. For a clean rerun in another environment, place the original file at `data/raw/customer_churn_a5.csv`.


## 1. Imports and Reproducibility

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

print("Libraries imported successfully.")
print("Random state:", RANDOM_STATE)


Libraries imported successfully.
Random state: 42


## 2. Data Loading and Initial Inspection


In [2]:
from pathlib import Path

candidate_paths = [
    Path("data/raw/customer_churn_a5.csv"),
    Path("../data/raw/customer_churn_a5.csv"),
    Path("customer_churn_a5.csv"),
    Path("/mnt/data/customer_churn_a5.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "customer_churn_a5.csv was not found. Expected data/raw/customer_churn_a5.csv "
        "or one of the supported fallback paths."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


Dataset path: customer_churn_a5.csv
Shape: (600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,churn,days_since_cancellation,final_bill_amount
0,20406,13,99.170000,Month-to-month,2,0,NaN,NaN
1,20531,24,66.770000,Month-to-month,1,0,NaN,NaN
2,20307,15,44.030000,One year,1,0,NaN,NaN
3,20391,3,61.080000,Two year,3,0,NaN,NaN
4,20147,8,106.720000,Two year,1,0,NaN,NaN


## 3. Dataset Columns

The legitimate predictors are `tenure_months`, `monthly_charges`, `contract_type`, and `support_calls`. The columns `days_since_cancellation` and `final_bill_amount` are target-leaked because they are only available after cancellation.


In [3]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [df[c].dtype for c in df.columns],
    "missing": [df[c].isna().sum() for c in df.columns],
    "unique": [df[c].nunique(dropna=False) for c in df.columns],
})
display(column_summary)


,column,dtype,missing,unique
0,customer_id,int64,0,600
1,tenure_months,int64,0,68
2,monthly_charges,float64,0,562
3,contract_type,object,0,3
4,support_calls,int64,0,9
5,churn,int64,0,2
6,days_since_cancellation,float64,451,75
7,final_bill_amount,float64,451,150


# 4.1 — Target Leakage

### Step 1 — Check the target-leaked cancellation feature


In [ ]:
non_null_days = df["days_since_cancellation"].notna().sum()
churn_one_rows = (df["churn"] == 1).sum()

print("Non-null days_since_cancellation:", non_null_days)
print("Rows where churn == 1:", churn_one_rows)
print("Counts exactly equal:", non_null_days == churn_one_rows)

assert non_null_days == churn_one_rows
print("CONFIRMED: every non-null days_since_cancellation value corresponds exactly to churn = 1.")


Non-null days_since_cancellation: 149
Rows where churn == 1: 149
Counts exactly equal: True
CONFIRMED: every non-null days_since_cancellation value corresponds exactly to churn = 1.


### Step 2 — Correlation of the leaked feature

`final_bill_amount` is filled with `-1` before calculating its correlation with `churn`. The result is compared with the legitimate `tenure_months` correlation.


In [5]:
final_bill_filled = df["final_bill_amount"].fillna(-1)

final_bill_corr = final_bill_filled.corr(df["churn"])
tenure_corr = df["tenure_months"].corr(df["churn"])

correlation_results = pd.DataFrame({
    "feature": ["final_bill_amount (missing -> -1)", "tenure_months"],
    "correlation_with_churn": [final_bill_corr, tenure_corr],
})
display(correlation_results)

print(f"final_bill_amount correlation: {final_bill_corr:.6f}")
print(f"tenure_months correlation:      {tenure_corr:.6f}")
print(f"Absolute correlation gap:       {abs(final_bill_corr) - abs(tenure_corr):.6f}")


,feature,correlation_with_churn
0,final_bill_amount (missing -> -1),0.924978
1,tenure_months,-0.220718


final_bill_amount correlation: 0.924978
tenure_months correlation:      -0.220718
Absolute correlation gap:       0.704260


### Step 3 — Model A: Legitimate features only

Model A uses only the four legitimate predictors. The split is performed **before** fitting preprocessing. A `ColumnTransformer` uses an imputer and scaler for numeric columns and one-hot encoding for `contract_type`, followed by Logistic Regression.


In [6]:
LEGIT_FEATURES = [
    "tenure_months",
    "monthly_charges",
    "contract_type",
    "support_calls",
]

NUMERIC_FEATURES = [
    "tenure_months",
    "monthly_charges",
    "support_calls",
]

CATEGORICAL_FEATURES = ["contract_type"]

X = df[LEGIT_FEATURES]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_A = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

model_A = Pipeline([
    ("preprocessor", preprocessor_A),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model_A.fit(X_train, y_train)

pred_A = model_A.predict(X_test)
prob_A = model_A.predict_proba(X_test)[:, 1]

accuracy_A = accuracy_score(y_test, pred_A)
auc_A = roc_auc_score(y_test, prob_A)

print(f"Model A test accuracy: {accuracy_A:.6f}")
print(f"Model A test ROC-AUC:  {auc_A:.6f}")


Model A test accuracy: 0.766667
Model A test ROC-AUC:  0.798519


### Step 4 — Model B: Legitimate + target-leaked features

Model B adds `days_since_cancellation` and `final_bill_amount`. Missing values in these leakage columns are replaced with `-1`.


In [7]:
LEAKAGE_FEATURES = [
    "days_since_cancellation",
    "final_bill_amount",
]

MODEL_B_FEATURES = LEGIT_FEATURES + LEAKAGE_FEATURES

X_B = df[MODEL_B_FEATURES].copy()
X_B[LEAKAGE_FEATURES] = X_B[LEAKAGE_FEATURES].fillna(-1)

X_B_train, X_B_test, y_B_train, y_B_test = train_test_split(
    X_B, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

preprocessor_B = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        NUMERIC_FEATURES + LEAKAGE_FEATURES
    ),
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        CATEGORICAL_FEATURES
    ),
])

model_B = Pipeline([
    ("preprocessor", preprocessor_B),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model_B.fit(X_B_train, y_B_train)

pred_B = model_B.predict(X_B_test)
prob_B = model_B.predict_proba(X_B_test)[:, 1]

accuracy_B = accuracy_score(y_B_test, pred_B)
auc_B = roc_auc_score(y_B_test, prob_B)

print(f"Model B test accuracy: {accuracy_B:.6f}")
print(f"Model B test ROC-AUC:  {auc_B:.6f}")


Model B test accuracy: 1.000000
Model B test ROC-AUC:  1.000000


### Step 5 — Model A vs Model B gap

A large improvement from adding the leaked variables is evidence that the model is learning information that would not be legitimately available at prediction time.


In [8]:
accuracy_gap = accuracy_B - accuracy_A
auc_gap = auc_B - auc_A

model_comparison = pd.DataFrame({
    "metric": ["Test Accuracy", "ROC-AUC"],
    "Model A (legitimate)": [accuracy_A, auc_A],
    "Model B (leaked)": [accuracy_B, auc_B],
    "B - A gap": [accuracy_gap, auc_gap],
})
display(model_comparison)

print(
    "Production explanation: Model B relies on fields that are outcomes of cancellation "
    "and therefore do not exist for a brand-new applicant before the prediction decision."
)
print(
    "If deployed to new applicants, those fields would be unavailable (or artificially filled), "
    "so Model B's validation performance would not represent real production performance."
)


,metric,Model A (legitimate),Model B (leaked),B - A gap
0,Test Accuracy,0.766667,1.000000,0.233333
1,ROC-AUC,0.798519,1.000000,0.201481


Production explanation: Model B relies on fields that are outcomes of cancellation and therefore do not exist for a brand-new applicant before the prediction decision.
If deployed to new applicants, those fields would be unavailable (or artificially filled), so Model B's validation performance would not represent real production performance.


### 4.1 Conclusion

**Target leakage:** the cancellation-related fields are only present when churn has already occurred, so they encode information that becomes known after the target event. Model B therefore receives information about the outcome itself and can achieve an unrealistically strong validation result.

For a new applicant who has not decided to cancel, `days_since_cancellation` and `final_bill_amount` cannot legitimately be known. A deployed Model B would therefore be unable to reproduce its validation-time performance and would not be a valid predictive model.


# 4.2 — Preprocessing Leakage

The three legitimate numeric features:

- `tenure_months`
- `monthly_charges`
- `support_calls`

The incorrect approach fits `StandardScaler` on the entire dataset before the train/test split. The correct approach splits first and fits the scaler only on the training data.


### Step 1 — Wrong order: fit scaler on the entire dataset, then split


In [9]:
scaler_full = StandardScaler()
scaler_full.fit(df[NUMERIC_FEATURES])

print("Scaler fitted on ENTIRE dataset")
print("mean_:", scaler_full.mean_)
print("scale_:", scaler_full.scale_)


Scaler fitted on ENTIRE dataset
mean_: [19.56166667 65.92201667  2.165     ]
scale_: [17.52454081 22.79931919  1.49814608]


### Step 2 — Correct order: split first, then fit scaler on training data only


In [10]:
scaler_train = StandardScaler()
scaler_train.fit(X_train[NUMERIC_FEATURES])

print("Scaler fitted on TRAINING SPLIT ONLY")
print("mean_:", scaler_train.mean_)
print("scale_:", scaler_train.scale_)


Scaler fitted on TRAINING SPLIT ONLY
mean_: [19.73958333 65.72710417  2.11458333]
scale_: [17.70077964 22.41337332  1.46820321]


### Step 3 — Difference between scaler means

The numeric difference below is calculated as:

`full-dataset scaler mean - training-only scaler mean`

Even if the values are numerically small, the principle matters because the test set is supposed to represent unseen future data. Letting test observations influence preprocessing parameters gives information from the evaluation data to the training procedure, making the evaluation less independent and potentially optimistic.


In [11]:
mean_difference = scaler_full.mean_ - scaler_train.mean_

mean_difference_table = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "full_dataset_mean": scaler_full.mean_,
    "training_only_mean": scaler_train.mean_,
    "difference": mean_difference,
})
display(mean_difference_table)

print(
    "Principle: preprocessing must be learned only from training data. "
    "Otherwise information from the held-out test set influences the transformation."
)
print(
    "The exact size of the difference is dataset-dependent; the leakage risk exists "
    "even when the numerical difference happens to be small."
)


,feature,full_dataset_mean,training_only_mean,difference
0,tenure_months,19.561667,19.739583,-0.177917
1,monthly_charges,65.922017,65.727104,0.194912
2,support_calls,2.165000,2.114583,0.050417


Principle: preprocessing must be learned only from training data. Otherwise information from the held-out test set influences the transformation.
The exact size of the difference is dataset-dependent; the leakage risk exists even when the numerical difference happens to be small.


### Step 4 — Correct preprocessing + model in one sklearn Pipeline

A single `sklearn.Pipeline` containing **imputer + scaler + model**, fitted only on the training split.

Because the legitimate numeric features contain no missing values in this dataset, the imputer does not change the values. Its presence still makes the pipeline explicit and deployable.


In [12]:
correct_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

correct_numeric_pipeline.fit(X_train[NUMERIC_FEATURES], y_train)

pipeline_scaler = correct_numeric_pipeline.named_steps["scaler"]

print("Pipeline scaler mean_:", pipeline_scaler.mean_)
print("Manual training scaler mean_:", scaler_train.mean_)
print()
print("Pipeline scaler scale_:", pipeline_scaler.scale_)
print("Manual training scaler scale_:", scaler_train.scale_)

means_match = np.allclose(pipeline_scaler.mean_, scaler_train.mean_, rtol=0, atol=1e-12)
scales_match = np.allclose(pipeline_scaler.scale_, scaler_train.scale_, rtol=0, atol=1e-12)

print()
print("Means match exactly within numerical tolerance:", means_match)
print("Scales match exactly within numerical tolerance:", scales_match)

assert means_match and scales_match
print("CONFIRMED: the pipeline reproduces the manual training-only scaler statistics.")


Pipeline scaler mean_: [19.73958333 65.72710417  2.11458333]
Manual training scaler mean_: [19.73958333 65.72710417  2.11458333]

Pipeline scaler scale_: [17.70077964 22.41337332  1.46820321]
Manual training scaler scale_: [17.70077964 22.41337332  1.46820321]

Means match exactly within numerical tolerance: True
Scales match exactly within numerical tolerance: True
CONFIRMED: the pipeline reproduces the manual training-only scaler statistics.


### 4.2 Conclusion

The wrong scaler uses information from all 600 observations, including observations that later become part of the test set. The correct scaler learns its parameters only from the training data, preserving the test set as unseen data.

The pipeline check confirms that the training-only scaler statistics are reproduced exactly by the `Pipeline`, which is the safer and deployable implementation.


# 4.3 — The Fix

The final model uses only legitimate features and performs the correct operation order:

1. Split into training and test sets.
2. Fit imputation/scaling/encoding only through the training data.
3. Train the model.
4. Evaluate on the untouched test set.


In [13]:
final_model = Pipeline([
    ("preprocessor", ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES
        ),
    ])),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

final_model.fit(X_train, y_train)

final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:, 1]

final_accuracy = accuracy_score(y_test, final_pred)
final_auc = roc_auc_score(y_test, final_prob)

print(f"Final fixed model test accuracy: {final_accuracy:.6f}")
print(f"Final fixed model test ROC-AUC:  {final_auc:.6f}")

print()
print("Accuracy matches Model A:", np.isclose(final_accuracy, accuracy_A, rtol=0, atol=1e-12))
print("ROC-AUC matches Model A:", np.isclose(final_auc, auc_A, rtol=0, atol=1e-12))

assert np.isclose(final_accuracy, accuracy_A, rtol=0, atol=1e-12)
assert np.isclose(final_auc, auc_A, rtol=0, atol=1e-12)

print("CONFIRMED: the final fixed model matches the honest Model A.")


Final fixed model test accuracy: 0.766667
Final fixed model test ROC-AUC:  0.798519

Accuracy matches Model A: True
ROC-AUC matches Model A: True
CONFIRMED: the final fixed model matches the honest Model A.


## Final Results Summary

The key results generated from the supplied dataset are summarized below.


In [14]:
final_summary = pd.DataFrame({
    "result": [
        "Non-null days_since_cancellation",
        "Rows with churn = 1",
        "final_bill_amount correlation with churn",
        "tenure_months correlation with churn",
        "Model A accuracy",
        "Model A ROC-AUC",
        "Model B accuracy",
        "Model B ROC-AUC",
        "Accuracy gap (B - A)",
        "ROC-AUC gap (B - A)",
        "Final fixed model accuracy",
        "Final fixed model ROC-AUC",
    ],
    "value": [
        non_null_days,
        churn_one_rows,
        final_bill_corr,
        tenure_corr,
        accuracy_A,
        auc_A,
        accuracy_B,
        auc_B,
        accuracy_gap,
        auc_gap,
        final_accuracy,
        final_auc,
    ]
})
display(final_summary)


,result,value
0,Non-null days_since_cancellation,149.000000
1,Rows with churn = 1,149.000000
2,final_bill_amount correlation with churn,0.924978
3,tenure_months correlation with churn,-0.220718
4,Model A accuracy,0.766667
5,Model A ROC-AUC,0.798519
6,Model B accuracy,1.000000
7,Model B ROC-AUC,1.000000
8,Accuracy gap (B - A),0.233333
9,ROC-AUC gap (B - A),0.201481


## Conclusion

### Target leakage
The dataset contains two features that become available only after churn: `days_since_cancellation` and `final_bill_amount`. Their inclusion gives the model access to post-outcome information, which creates an unrealistically strong validation result.

### Preprocessing leakage
A scaler fitted on the full dataset learns statistics from observations that are later used for testing. The correct approach is to split first and fit preprocessing only on training data.

### Final fix
The final deployable model uses only legitimate predictors and a pipeline that keeps preprocessing inside the training workflow. Its performance matches Model A, confirming that the honest baseline and the final fixed implementation are consistent.


# Bonus — Cross-Validation Leakage

Isolates the numeric preprocessing step. The incorrect version scales the entire dataset before 5-fold cross-validation. The correct version places `StandardScaler` inside the pipeline passed to `cross_val_score`, so each fold learns its own scaler from its training portion only.

A small difference does **not** mean the leakage principle is unimportant; it can simply mean that the dataset is not very sensitive to the leaked scaling statistics.


In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Incorrect: scaler fitted before cross-validation
full_scaler_for_cv = StandardScaler()
X_scaled_full = full_scaler_for_cv.fit_transform(df[NUMERIC_FEATURES])

wrong_cv_scores = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_scaled_full,
    y,
    cv=cv,
    scoring="roc_auc"
)

# Correct: scaler fitted separately inside each CV training fold
proper_cv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

proper_cv_scores = cross_val_score(
    proper_cv_pipeline,
    df[NUMERIC_FEATURES],
    y,
    cv=cv,
    scoring="roc_auc"
)

bonus_results = pd.DataFrame({
    "fold": range(1, 6),
    "incorrect_pre_scaled_auc": wrong_cv_scores,
    "proper_pipeline_auc": proper_cv_scores,
    "difference": wrong_cv_scores - proper_cv_scores,
})
display(bonus_results)

print(f"Incorrect pre-scaled CV mean ROC-AUC: {wrong_cv_scores.mean():.6f}")
print(f"Proper Pipeline CV mean ROC-AUC:       {proper_cv_scores.mean():.6f}")
print(f"Mean difference:                       {(wrong_cv_scores.mean() - proper_cv_scores.mean()):.6f}")


,fold,incorrect_pre_scaled_auc,proper_pipeline_auc,difference
0,1,0.666540,0.666540,0.000000
1,2,0.712593,0.712593,0.000000
2,3,0.714815,0.714815,0.000000
3,4,0.650000,0.649259,0.000741
4,5,0.620000,0.620000,0.000000


Incorrect pre-scaled CV mean ROC-AUC: 0.672790
Proper Pipeline CV mean ROC-AUC:       0.672641
Mean difference:                       0.000148


## Interpretation

The correct pattern is to place preprocessing inside the estimator pipeline supplied to `cross_val_score`. This ensures every cross-validation training fold learns preprocessing parameters without seeing its validation fold.

If the two scores are very close on this dataset, that is still consistent with the assignment's warning: leakage is a methodological risk even when its measured effect happens to be small.
